In [24]:
from imitation.scripts.generate_ranked_trajectories import generate_policies_and_videos, generate_trajectories, record_trajectory_video 
import numpy as np

ENV_NAME = "CartPole-v1"
ITERATION_STEP = 1000
NUM_POLICIES = 10

video_entries, policy_entries = generate_policies_and_videos(NUM_POLICIES, ITERATION_STEP, ENV_NAME)
trajectory_entries = generate_trajectories(policy_entries[0:len(policy_entries)//2], ENV_NAME, no_trajectories=10)

for i in range(0, len(trajectory_entries)):
    print(trajectory_entries[i]["trajectory"].rews.shape) # why is reward capped at 500 ? 

print(len(trajectory_entries))


ppo_cartpole_1000 already exists.
ppo_cartpole_2000 already exists.
ppo_cartpole_3000 already exists.
ppo_cartpole_4000 already exists.
ppo_cartpole_5000 already exists.
ppo_cartpole_6000 already exists.
ppo_cartpole_7000 already exists.
ppo_cartpole_8000 already exists.
ppo_cartpole_9000 already exists.
ppo_cartpole_10000 already exists.
(62,)
(42,)
(53,)
(44,)
(42,)
(52,)
(45,)
(39,)
(49,)
(64,)
(115,)
(146,)
(144,)
(124,)
(151,)
(134,)
(125,)
(128,)
(116,)
(133,)
(206,)
(270,)
(157,)
(265,)
(253,)
(141,)
(184,)
(267,)
(213,)
(273,)
(500,)
(182,)
(288,)
(321,)
(216,)
(176,)
(500,)
(500,)
(167,)
(500,)
(500,)
(500,)
(500,)
(179,)
(211,)
(212,)
(204,)
(245,)
(210,)
(486,)
50


In [26]:
def make_fragments(traj, frag_len=25, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    T = len(traj.obs) - 1  # last obs has no action
    if T < frag_len: return []
    starts = rng.integers(0, T - frag_len + 1, size=max(1, T // frag_len))
    return [(traj.obs[s:s+frag_len], traj.rews) for s in starts]

def make_ranked_pairs(trajs, n_pairs=20000, frag_len=25, rng=None, tie_margin=0.0):
    rng = np.random.default_rng() if rng is None else rng
    # flatten all fragments
    frags = []
    for tr in trajs:
        trj = tr["trajectory"]
        frags += make_fragments(trj, frag_len, rng)
        print(len(frags))
    frags = [(o.astype(np.float32), r.astype(np.float32)) for o,r in frags]
    # sample pairs + label by true return
    pairs = []
    for _ in range(min(n_pairs, len(frags)**2)):
        i, j = rng.integers(0, len(frags), size=2)
        (o1, r1), (o2, r2) = frags[i], frags[j]
        g1, g2 = float(r1.sum()), float(r2.sum())
        if abs(g1-g2) <= tie_margin: continue
        y = 1.0 if g1 > g2 else 0.0
        pairs.append((o1, o2, y))
    return pairs

In [27]:
# Building ranked pairs as a learning dataset
pairs = []
for i, t1 in enumerate(trajectory_entries):
    for j, t2 in enumerate(trajectory_entries):
        if i >= j: continue
        if sum(t1["trajectory"].rews) == sum(t2["trajectory"].rews): continue 
        y = 1.0 if sum(t1["trajectory"].rews) > sum(t2["trajectory"].rews) else 0.0
        pairs.append((t1["trajectory"].obs, t2["trajectory"].obs, y))

print(len(pairs))

1203


In [28]:
pairs = []
pairs = make_ranked_pairs(trajectory_entries)
print(len(pairs))

2
3
5
6
7
9
10
11
12
14
18
23
28
32
38
43
48
53
57
62
70
80
86
96
106
111
118
128
136
146
166
173
184
196
204
211
231
251
257
277
297
317
337
344
352
360
368
377
385
404
17335


In [ ]:

from torch.utils.data import Dataset, DataLoader
import torch

def collate_pad(batch):
    # pad sequences to same T (left pad=False → right pad with zeros)
    o1s, o2s, ys = zip(*batch)
    T1 = max(x.shape[0] for x in o1s); T2 = max(x.shape[0] for x in o2s)
    def pad(stack, T):
        D = stack[0].shape[1]
        out = torch.zeros(len(stack), T, D)
        for i,x in enumerate(stack):
            out[i,:x.shape[0]] = x
        return out
    return pad(o1s, T1), pad(o2s, T2), torch.stack(ys)

class PrefPairsDS(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        o1, o2, y = self.pairs[i]               # (T, obs_dim), (T, obs_dim), scalar
        return torch.from_numpy(o1), torch.from_numpy(o2), torch.tensor(y, dtype=torch.float32)


train_loader = DataLoader(PrefPairsDS(pairs), batch_size=64, shuffle=True, collate_fn=collate_pad)

In [31]:
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

obs_dim = gym.make(ENV_NAME).observation_space.shape[0]

class RewardMLP(nn.Module):
    def __init__(self, obs_dim, hid=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hid), nn.Tanh(),
            nn.Linear(hid, hid), nn.Tanh(),
            nn.Linear(hid, 1)
        )
    def forward(self, x):            # x: (B, D)
        return self.net(x).squeeze(-1)

reward_net = RewardMLP(obs_dim)
opt = torch.optim.Adam(reward_net.parameters(), lr=3e-4)
device = torch.device("cpu"); reward_net.to(device)

def pred_frag_return(obs_seq):       # (B, T, D) -> (B,)
    B,T,D = obs_seq.shape
    r = reward_net(obs_seq.reshape(B*T, D)).view(B,T).sum(dim=1)
    return r

def bt_loss(r1, r2, y):              # y in {0,1}
    return F.binary_cross_entropy_with_logits(r1 - r2, y)



In [ ]:

for step, (o1, o2, y) in enumerate(train_loader):
    o1, o2, y = o1.to(device), o2.to(device), y.to(device)
    T = min(o1.shape[1], o2.shape[1]); o1, o2 = o1[:,:T], o2[:,:T]
    r1, r2 = pred_frag_return(o1), pred_frag_return(o2)
    loss = bt_loss(r1, r2, y)
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(reward_net.parameters(), 5.0)
    opt.step()
    print(f"{step:05d}  loss={loss.item():.4f}")

00000  loss=1.1616
00001  loss=0.8957
00002  loss=0.7538
00003  loss=0.6480
00004  loss=0.7592
00005  loss=0.7841
00006  loss=0.6261
00007  loss=0.6426
00008  loss=0.6616
00009  loss=0.5892
00010  loss=0.6973
00011  loss=0.6920
00012  loss=0.7027
00013  loss=0.7848
00014  loss=0.6784
00015  loss=0.6550
00016  loss=0.6001
00017  loss=0.7302
00018  loss=0.7269
00019  loss=0.5415
00020  loss=0.6277
00021  loss=0.5692
00022  loss=0.6518
00023  loss=0.6305
00024  loss=0.6152
00025  loss=0.6030
00026  loss=0.6104
00027  loss=0.6495
00028  loss=0.6041
00029  loss=0.6018
00030  loss=0.6213
00031  loss=0.6076
00032  loss=0.6049
00033  loss=0.6085
00034  loss=0.6801
00035  loss=0.5403
00036  loss=0.5809
00037  loss=0.6580
00038  loss=0.5943
00039  loss=0.5867
00040  loss=0.5423
00041  loss=0.5500
00042  loss=0.5732
00043  loss=0.5507
00044  loss=0.6312
00045  loss=0.6269
00046  loss=0.6358
00047  loss=0.5664
00048  loss=0.5344
00049  loss=0.5124
00050  loss=0.4952
00051  loss=0.4854
00052  loss=